In [1]:
#Jan Poreba
import pandas as pd
from ucimlrepo import fetch_ucirepo


adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets


df = X.copy()
df['income'] = y

print(f"Początkowa liczba rekordów: {len(df)}")
print(df.head(3))
print("\n" + "="*80 + "\n")





Początkowa liczba rekordów: 48842
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   

       marital-status         occupation   relationship   race   sex  \
0       Never-married       Adm-clerical  Not-in-family  White  Male   
1  Married-civ-spouse    Exec-managerial        Husband  White  Male   
2            Divorced  Handlers-cleaners  Not-in-family  White  Male   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0             0              13  United-States  <=50K  
2             0             0              40  United-States  <=50K  




In [2]:
# Pseudonimizacja i wybór atrybutów
import uuid

# Nadanie anonimowego losowego identyfikatora (pierwsze 8 znaków z UUID)
df['anon_ID'] = [str(uuid.uuid4())[:8] for _ in range(len(df))]

# quasi-identyfikatory (Q)
Q = ['age', 'education', 'sex', 'race']
# Atrybut wrażliwy
sensitive_attr = 'income'

# Tworzymy nowy zbiór roboczy zawierający tylko potrzebne kolumny
df_anon = df[['anon_ID'] + Q + [sensitive_attr]].copy()
print(df_anon.head(3))
print("\n" + "="*80 + "\n")

    anon_ID  age  education   sex   race income
0  edda3d77   39  Bachelors  Male  White  <=50K
1  1d54fd22   50  Bachelors  Male  White  <=50K
2  da72797a   38    HS-grad  Male  White  <=50K




In [3]:

# Generalizacja
# Aby trudniej było zidentyfikować jednostkę, zmniejszamy precyzję danych.
# Zamiast dokładnego wieku podamy przedziały
# Edukację pogrupujemy w szersze kategorie.
def generalize_age(age):
    if pd.isna(age): return 'Unknown'
    decade = (int(age) // 10) * 10
    return f"{decade}-{decade+9}"

def generalize_education(edu):
    higher_edu = ['Bachelors', 'Masters', 'Doctorate', 'Prof-school']
    if edu in higher_edu:
        return 'Higher Education'
    else:
        return 'Up to College'

df_anon['age'] = df_anon['age'].apply(generalize_age)
df_anon['education'] = df_anon['education'].apply(generalize_education)

print("Dane po zastosowaniu Generalizacji (wiek w przedziałach, ogólniejsze wykształcenie):")
print(df_anon.head(5))
print("\n" + "="*80 + "\n")

Dane po zastosowaniu Generalizacji (wiek w przedziałach, ogólniejsze wykształcenie):
    anon_ID    age         education     sex   race income
0  edda3d77  30-39  Higher Education    Male  White  <=50K
1  1d54fd22  50-59  Higher Education    Male  White  <=50K
2  da72797a  30-39     Up to College    Male  White  <=50K
3  f3f4327c  50-59     Up to College    Male  Black  <=50K
4  febd07f4  20-29  Higher Education  Female  Black  <=50K




In [4]:
# Generowanie wszystkich klas równoważności

def generate_equivalence_classes(dataframe, quasi_identifiers):
    """
    Generuje wszystkie klasy równoważności dla zadanego zbioru danych.
    Zwraca słownik, gdzie:
    - klucz: krotka z kombinacją wartości quasi-identyfikatorów (np. wiek, edukacja, płeć, rasa)
    - wartość: DataFrame zawierający wszystkie rekordy przypisane do tej klasy.
    """
    classes = {}
    for key, group in dataframe.groupby(quasi_identifiers):
        classes[key] = group
    return classes

# Wywołanie nowej metody na anonimizowanym zbiorze
all_eq_classes = generate_equivalence_classes(df_anon, Q)
print(f"Zidentyfikowano łącznie {len(all_eq_classes)} unikalnych klas równoważności w zbiorze przed nałożeniem k-anonimowości.")

# Wyświetlenie informacji o 3 przykładowych klasach
print("Podgląd rozmiarów 3 pierwszych klas równoważności:")
for i, (eq_key, eq_group) in enumerate(all_eq_classes.items()):
    if i >= 3:
        break
    print(f" -> Klasa Q {eq_key}: {len(eq_group)} rekordów")
print("\n" + "="*80 + "\n")

Zidentyfikowano łącznie 145 unikalnych klas równoważności w zbiorze przed nałożeniem k-anonimowości.
Podgląd rozmiarów 3 pierwszych klas równoważności:
 -> Klasa Q ('10-19', 'Higher Education', 'Female', 'Black'): 1 rekordów
 -> Klasa Q ('10-19', 'Higher Education', 'Female', 'White'): 2 rekordów
 -> Klasa Q ('10-19', 'Up to College', 'Female', 'Amer-Indian-Eskimo'): 15 rekordów




In [5]:
# Osiągnięcie k-anonimowości
# Teraz sprawdzamy klasy równoważności (EC). Klasa równoważności to grupa
# rekordów o identycznych wartościach dla wszystkich quasi-identyfikatorów Q.
# Jeśli jakaś klasa liczy mniej niż k rekordów, usuwamy te rekordy,
# aby zagwarantować, że każdy w zbiorze chowa się w grupie liczacej co najmniej
# k identycznych osób.

k = 10  # Ustawiamy wartość parametru k

# 1. Grupujemy po quasi-identyfikatorach i liczymy wielkość każdej klasy równoważności
ec_sizes = df_anon.groupby(Q).size().reset_index(name='ec_size')

# 2. Łączymy rozmiary klas ze zbiorem danych
df_k_anon = pd.merge(df_anon, ec_sizes, on=Q)

# 3. Odrzucamy rekordy, dla których rozmiar klasy jest mniejszy niż k
df_final = df_k_anon[df_k_anon['ec_size'] >= k].copy()

# usuwamy kolumnę pomocniczą z rozmiarem klasy
df_final = df_final.drop(columns=['ec_size'])

records_lost = len(df_anon) - len(df_final)
print(f"--- PODSUMOWANIE k-ANONIMIZACJI ---")
print(f"Ustawiono parametr k = {k}")
print(f"Liczba rekordów po anonimizacji: {len(df_final)}")
print(f"Odrzucono (supresja) {records_lost} rekordów, aby spełnić warunek k-anonimowości.\n")


sample_group = df_final[(df_final['age'] == '30-39') &
                        (df_final['sex'] == 'Female') &
                        (df_final['race'] == 'White') &
                        (df_final['education'] == 'Higher Education')]

print(sample_group.head(10))

--- PODSUMOWANIE k-ANONIMIZACJI ---
Ustawiono parametr k = 10
Liczba rekordów po anonimizacji: 48667
Odrzucono (supresja) 175 rekordów, aby spełnić warunek k-anonimowości.

      anon_ID    age         education     sex   race income
5    e87c2e62  30-39  Higher Education  Female  White  <=50K
8    ac0a22a8  30-39  Higher Education  Female  White   >50K
188  80f8368d  30-39  Higher Education  Female  White  <=50K
260  f9cf4d9a  30-39  Higher Education  Female  White  <=50K
268  f4e5bdc6  30-39  Higher Education  Female  White  <=50K
296  5672da47  30-39  Higher Education  Female  White  <=50K
422  8630bee5  30-39  Higher Education  Female  White   >50K
469  8444a725  30-39  Higher Education  Female  White   >50K
538  0b0e0962  30-39  Higher Education  Female  White  <=50K
698  53a5d1d0  30-39  Higher Education  Female  White  <=50K
      anon_ID    age         education     sex   race income
5    e87c2e62  30-39  Higher Education  Female  White  <=50K
8    ac0a22a8  30-39  Higher Educa

In [ ]:
print(df_anon.head(5))
print("\n")

print("="*80)
print("4. ANALIZA KLAS RÓWNOWAŻNOŚCI (PRZED K-ANONIMIZACJĄ)")
print("="*80)

def generate_equivalence_classes(dataframe, quasi_identifiers):
    classes = {}
    for key, group in dataframe.groupby(quasi_identifiers):
        classes[key] = group
    return classes

all_eq_classes = generate_equivalence_classes(df_anon, Q)
print(f"Zidentyfikowano łącznie {len(all_eq_classes)} unikalnych klas równoważności w zbiorze przed nałożeniem k-anonimowości.")

print("Podgląd rozmiarów 3 pierwszych klas równoważności:")
for i, (eq_key, eq_group) in enumerate(all_eq_classes.items()):
    if i >= 3: break
    print(f" -> Klasa Q {eq_key}: {len(eq_group)} rekordów")
print("\n")

print("="*80)
print("5. OSIĄGNIĘCIE K-ANONIMOWOŚCI (DLA K=10)")
print("="*80)

k_10 = 10

ec_sizes_10 = df_anon.groupby(Q).size().reset_index(name='ec_size')
df_k_anon_10 = pd.merge(df_anon, ec_sizes_10, on=Q)
df_final_10 = df_k_anon_10[df_k_anon_10['ec_size'] >= k_10].copy()
df_final_10 = df_final_10.drop(columns=['ec_size'])

records_lost_10 = len(df_anon) - len(df_final_10)
print(f"Ustawiono parametr k = {k_10}")
print(f"Liczba rekordów po anonimizacji: {len(df_final_10)}")
print(f"Odrzucono (supresja) {records_lost_10} rekordów, aby spełnić warunek k-anonimowości.\n")

sample_group_10 = df_final_10[(df_final_10['age'] == '30-39') &
                                (df_final_10['sex'] == 'Female') &
                                (df_final_10['race'] == 'White') &
                                (df_final_10['education'] == 'Higher Education')]
print(sample_group_10.head(10))
print("\n")

print("="*80)
print("6. OSIĄGNIĘCIE K-ANONIMOWOŚCI (DLA K=3)")
print("="*80)

k = 3

ec_sizes = df_anon.groupby(Q).size().reset_index(name='ec_size')
df_k_anon = pd.merge(df_anon, ec_sizes, on=Q)
df_final = df_k_anon[df_k_anon['ec_size'] >= k].copy()
df_final = df_final.drop(columns=['ec_size'])

records_lost = len(df_anon) - len(df_final)
print(f"Ustawiono parametr k = {k}")
print(f"Liczba rekordów po anonimizacji: {len(df_final)}")
print(f"Odrzucono (supresja) {records_lost} rekordów, aby spełnić warunek k-anonimowości.\n")

sample_group = df_final[(df_final['age'] == '30-39') &
                        (df_final['sex'] == 'Female') &
                        (df_final['race'] == 'White') &
                        (df_final['education'] == 'Higher Education')]
print(sample_group.head(10))
print("\n")

print("="*80)
print("7. ROZBUDOWA: ODPOWIEDŹ NA PYTANIE KOŃCOWE Z NOTATNIKA")
print("="*80)

print("--- TEST 1: Globalne sprawdzenie zachowania k-anonimowości ---")
min_class_size = df_final.groupby(Q).size().min()
print(f"Minimalny rozmiar klasy równoważności w zanonimizowanym zbiorze to: {min_class_size}")

if min_class_size >= k:
    print(f"Zbiór spełnia warunek k-anonimowości dla k={k}!")
    print(f"Odpowiedź: NIE jesteśmy w stanie wskazać jednoznacznie na jedną osobę na podstawie cech {Q}.")
    print(f"Każda osoba dzieli swoje cechy z co najmniej {k-1} innymi osobami w zbiorze.")
else:
    print(f"\nBŁĄD: Zbiór NIE spełnia warunku k-anonimowości dla k={k}!")
print("\n")

    anon_ID    age         education     sex   race income
0  edda3d77  30-39  Higher Education    Male  White  <=50K
1  1d54fd22  50-59  Higher Education    Male  White  <=50K
2  da72797a  30-39     Up to College    Male  White  <=50K
3  f3f4327c  50-59     Up to College    Male  Black  <=50K
4  febd07f4  20-29  Higher Education  Female  Black  <=50K


4. ANALIZA KLAS RÓWNOWAŻNOŚCI (PRZED K-ANONIMIZACJĄ)
Zidentyfikowano łącznie 145 unikalnych klas równoważności w zbiorze przed nałożeniem k-anonimowości.
Podgląd rozmiarów 3 pierwszych klas równoważności:
 -> Klasa Q ('10-19', 'Higher Education', 'Female', 'Black'): 1 rekordów
 -> Klasa Q ('10-19', 'Higher Education', 'Female', 'White'): 2 rekordów
 -> Klasa Q ('10-19', 'Up to College', 'Female', 'Amer-Indian-Eskimo'): 15 rekordów


5. OSIĄGNIĘCIE K-ANONIMOWOŚCI (DLA K=10)
Ustawiono parametr k = 10
Liczba rekordów po anonimizacji: 48667
Odrzucono (supresja) 175 rekordów, aby spełnić warunek k-anonimowości.

      anon_ID    age         